In [1]:
import pandas as pd
import time
import os

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))

Dataset shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  


In [2]:
# Define paths
csv_path = "../data/processed/titanic.csv"
parquet_path = "../data/processed/titanic.parquet"
json_path = "../data/processed/titanic.json"

# Save in all 3 formats
df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path)  # requires pyarrow: pip install pyarrow
df.to_json(json_path, orient="records")

print("All files saved ✅")

All files saved ✅


In [3]:
# --- File Size Comparison ---
files = {
    "CSV":     csv_path,
    "Parquet": parquet_path,
    "JSON":    json_path
}

print("=" * 40)
print(f"{'Format':<10} {'Size (KB)':>10}")
print("=" * 40)

for fmt, path in files.items():
    size_kb = os.path.getsize(path) / 1024
    print(f"{fmt:<10} {size_kb:>10.2f} KB")

print("=" * 40)

Format      Size (KB)
CSV             61.42 KB
Parquet         40.80 KB
JSON           161.80 KB


In [4]:
import numpy as np

# Repeat the titanic data 1000x to simulate ~900k rows
df_large = pd.concat([df] * 1000, ignore_index=True)
print(f"Large dataset shape: {df_large.shape}")

# Save all formats
df_large.to_csv("../data/processed/titanic_large.csv", index=False)
df_large.to_parquet("../data/processed/titanic_large.parquet")
df_large.to_json("../data/processed/titanic_large.json", orient="records")

# Compare sizes
files_large = {
    "CSV":     "../data/processed/titanic_large.csv",
    "Parquet": "../data/processed/titanic_large.parquet",
    "JSON":    "../data/processed/titanic_large.json"
}

print("\n" + "=" * 40)
print(f"{'Format':<10} {'Size (MB)':>10}")
print("=" * 40)

for fmt, path in files_large.items():
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{fmt:<10} {size_mb:>10.2f} MB")

print("=" * 40)

Large dataset shape: (891000, 12)

Format      Size (MB)
CSV             59.90 MB
Parquet          3.76 MB
JSON           158.01 MB


In [5]:
# --- Read Speed Comparison ---
import time

runs = 3  # average over 3 runs for accuracy

def avg_read_time(read_fn, label):
    times = []
    for _ in range(runs):
        start = time.time()
        read_fn()
        times.append(time.time() - start)
    avg = sum(times) / runs
    print(f"{label:<10} avg read time: {avg:.4f} seconds")

print("=" * 45)
print(f"Reading {df_large.shape[0]:,} rows, {runs} runs each")
print("=" * 45)

avg_read_time(
    lambda: pd.read_csv("../data/processed/titanic_large.csv"),
    "CSV"
)
avg_read_time(
    lambda: pd.read_parquet("../data/processed/titanic_large.parquet"),
    "Parquet"
)

print("=" * 45)

Reading 891,000 rows, 3 runs each
CSV        avg read time: 1.2148 seconds
Parquet    avg read time: 0.0873 seconds


In [6]:
# --- Column Selection: Where Parquet Really Wins ---
# Real pipelines rarely need ALL columns
# They usually select 2-3 columns for a specific transformation

print("=" * 55)
print("Reading ONLY 2 columns (Survived, Fare) from 891k rows")
print("=" * 55)

avg_read_time(
    lambda: pd.read_csv(
        "../data/processed/titanic_large.csv",
        usecols=["Survived", "Fare"]
    ),
    "CSV"
)
avg_read_time(
    lambda: pd.read_parquet(
        "../data/processed/titanic_large.parquet",
        columns=["Survived", "Fare"]
    ),
    "Parquet"
)

print("=" * 55)
print("\nWith ALL 12 columns (from Step 5):")
print(f"  CSV     1.2148s")
print(f"  Parquet 0.0873s")

Reading ONLY 2 columns (Survived, Fare) from 891k rows
CSV        avg read time: 0.4537 seconds
Parquet    avg read time: 0.0115 seconds

With ALL 12 columns (from Step 5):
  CSV     1.2148s
  Parquet 0.0873s


# File Formats Comparison
**Dataset:** Titanic (891 rows → scaled to 891,000 rows)
**Date:** 2026-03-09

## Results Summary

| Format  | Size (891k rows) | Read Time (all cols) | Read Time (2 cols) |
|---------|-----------------|---------------------|-------------------|
| CSV     | 59.90 MB        | 1.2148s             | 0.4537s           |
| Parquet | 3.76 MB         | 0.0873s             | 0.0115s           |
| JSON    | 158.01 MB       | N/A                 | N/A               |

## Key Takeaways
- Parquet is 16x smaller than CSV at scale
- Parquet is 14x faster reading all columns
- Parquet is 39x faster reading 2 columns — columnar storage skips unneeded columns entirely
- JSON is for API responses, not analytics storage
- Use CSV only for small configs, seeds, or sharing with non-engineers